# Module 3: Macro Anomalies (Aug 27 Morocco & Aug 16 Peak), Weekly Seasonality, Corridors & Simpson's Paradox
**SQL Reference:** [`sql/03_anomalies_and_corridors.sql`](../sql/03_anomalies_and_corridors.sql) | **Visual Charts:** [`figures/anomaly_timeseries.png`](../figures/anomaly_timeseries.png), [`figures/04_corridors_archetypes_simpson.png`](../figures/04_corridors_archetypes_simpson.png)

## Executive Summary (`H3` Confirmed, `H6` Rejected)
1. **Aug 27 Morocco Domestic Viral Surge (`H3` Confirmed):** On Aug 27, iOS sessions spiked to `10,109` while observed iOS CR crashed to **`5.64%`** (`570` buyers). Country decomposition isolates **`5,134` Moroccan (`MA`) iOS sessions** (`50.79%` of global iOS traffic that day) converting at **`0.10%`** (`5` buyers). Across all `5,787` Moroccan sessions on Aug 27, `5,134` session-destination pairs were logged in `events`, of which **`5,019` (`97.8%`) searched for local `MA` data** (seeking cheap domestic cellular data rather than outbound travel roaming). Excluding Morocco restores Aug 27 iOS CR to **`11.36%`** (`565 / 4,975`).
2. **Aug 16 Global Peak & Weekly Seasonality (`1_DOW` Verified):** Friday Aug 16 represents the 27-day demand peak (**`1,130` orders / `1,105` buyers / `$16,240.64` revenue / `14.83%` iOS CR**). Across the 27-day period, **Fridays (`1.083x` index, `11.10%` CR)** and **Saturdays (`1.074x` index, `10.89%` CR)** are the true conversion and order peaks, whereas **Wednesdays (`0.922x` index, `833.0` buyers/day)** represent the weekly demand trough.
3. **Buyer Archetypes & In-Destination Purchases:** Regional Cross-Border drives `62.03%` of revenue (`$222,439.80`), **In-Destination Purchases (`geo_country = destination`) drive `22.89%` (`$82,100.94`, `5,939` orders — Airport Wi-Fi / Local Backup SIMs)**, and Long-Haul Transatlantic (`US/CA -> Europe`) commands a **`+25.0%` AOV premium (`$17.28`)**.
4. **Simpson's Paradox Check (`H6` Rejected):** iOS outperforms Compatible Android in **10 of the top 11 markets** (`+3.07 pp` in US, `+5.93 pp` in CA, `+7.02 pp` in RO), proving the `+2.84 pp` global gap is structural to the Android UX/promo journey rather than geographic mix.


In [1]:
import pandas as pd

# Verified BigQuery Output from sql/03_anomalies_and_corridors.sql (Query 3.3: Weekly DOW Seasonality)
dow_df = pd.DataFrame([
    {"Day": "1_Sunday (n=4)", "Avg_Sessions": 9266.5, "Avg_Buyers": 913.0, "Avg_Revenue_USD": 13145.26, "Session_CR_Pct": 9.85, "Order_Index": 1.011},
    {"Day": "2_Monday (n=4)", "Avg_Sessions": 9642.5, "Avg_Buyers": 895.0, "Avg_Revenue_USD": 13416.24, "Session_CR_Pct": 9.28, "Order_Index": 0.991},
    {"Day": "3_Tuesday_Raw (n=4)", "Avg_Sessions": 10355.5, "Avg_Buyers": 858.8, "Avg_Revenue_USD": 12760.99, "Session_CR_Pct": 8.29, "Order_Index": 0.951},
    {"Day": "3_Tuesday_Ex_Aug27 (n=3)", "Avg_Sessions": 8713.7, "Avg_Buyers": 845.7, "Avg_Revenue_USD": 12457.42, "Session_CR_Pct": 9.71, "Order_Index": 0.937},
    {"Day": "4_Wednesday_Trough (n=3)", "Avg_Sessions": 8752.0, "Avg_Buyers": 833.0, "Avg_Revenue_USD": 12449.33, "Session_CR_Pct": 9.52, "Order_Index": 0.922},
    {"Day": "5_Thursday (n=4)", "Avg_Sessions": 8870.8, "Avg_Buyers": 876.0, "Avg_Revenue_USD": 12757.69, "Session_CR_Pct": 9.88, "Order_Index": 0.970},
    {"Day": "6_Friday_Peak (n=4)", "Avg_Sessions": 8813.0, "Avg_Buyers": 978.0, "Avg_Revenue_USD": 14119.35, "Session_CR_Pct": 11.10, "Order_Index": 1.083},
    {"Day": "7_Saturday_Peak (n=4)", "Avg_Sessions": 8907.3, "Avg_Buyers": 969.8, "Avg_Revenue_USD": 14119.94, "Session_CR_Pct": 10.89, "Order_Index": 1.074}
])
print(dow_df.to_string(index=False))


Day  Avg_Sessions  Avg_Buyers  Avg_Revenue_USD  Session_CR_Pct  Order_Index
          1_Sunday (n=4)        9266.5       913.0         13145.26            9.85        1.011
          2_Monday (n=4)        9642.5       895.0         13416.24            9.28        0.991
     3_Tuesday_Raw (n=4)       10355.5       858.8         12760.99            8.29        0.951
3_Tuesday_Ex_Aug27 (n=3)        8713.7       845.7         12457.42            9.71        0.937
4_Wednesday_Trough (n=3)        8752.0       833.0         12449.33            9.52        0.922
        5_Thursday (n=4)        8870.8       876.0         12757.69            9.88        0.970
     6_Friday_Peak (n=4)        8813.0       978.0         14119.35           11.10        1.083
   7_Saturday_Peak (n=4)        8907.3       969.8         14119.94           10.89        1.074


In [2]:
# Verified BigQuery Output from sql/03_anomalies_and_corridors.sql (Query 3.5 & 3.6): Archetypes & Simpson's Paradox
archetypes_df = pd.DataFrame([
    {"Archetype": "1. Regional Cross-Border Travel", "Paid_Orders": 16034, "Share_Orders_Pct": 63.80, "Revenue_USD": 222439.80, "Share_Rev_Pct": 62.03, "AOV_USD": 13.87},
    {"Archetype": "2. In-Destination Purchase (Airport Wi-Fi / Local Backup)", "Paid_Orders": 5939, "Share_Orders_Pct": 23.63, "Revenue_USD": 82100.94, "Share_Rev_Pct": 22.89, "AOV_USD": 13.82},
    {"Archetype": "3. Long-Haul Transatlantic (US/CA -> Europe)", "Paid_Orders": 3026, "Share_Orders_Pct": 12.04, "Revenue_USD": 52285.60, "Share_Rev_Pct": 14.58, "AOV_USD": 17.28},
    {"Archetype": "4. Accidental Domestic (Morocco Viral Surge)", "Paid_Orders": 134, "Share_Orders_Pct": 0.53, "Revenue_USD": 1799.51, "Share_Rev_Pct": 0.50, "AOV_USD": 13.43}
])
print(archetypes_df.to_string(index=False))


Archetype  Paid_Orders  Share_Orders_Pct  Revenue_USD  Share_Rev_Pct  AOV_USD
                          1. Regional Cross-Border Travel        16034             63.80    222439.80          62.03    13.87
2. In-Destination Purchase (Airport Wi-Fi / Local Backup)         5939             23.63     82100.94          22.89    13.82
             3. Long-Haul Transatlantic (US/CA -> Europe)         3026             12.04     52285.60          14.58    17.28
             4. Accidental Domestic (Morocco Viral Surge)          134              0.53      1799.51           0.50    13.43
